# Семинар 9. Seaborn и Plotly: распределения, связи, множественные графики

Ноутбук занятия: материал разбирается вместе с преподавателем, ячейки выполняются сверху вниз.

- Переименуйте файл в формате `Группа-Фамилия-Имя-seminar-09.ipynb`, например `2MP9-Ivanov-Ivan-seminar-09.ipynb`.
- Ноутбук читает файлы `datasets/legal/dtp-bryansk.csv` и `datasets/legal/dtp-bryansk-participants.csv` из репозитория курса: путь в ячейке задан от папки `notebooks/`, поэтому папки `notebooks/` и `datasets/` должны лежать рядом, как в репозитории. Файлы сохраняются в папку `notebooks/img/`, она создаётся первой ячейкой и в репозиторий не попадает.
- Упражнения к занятию лежат в папке `exercises/`: листок `seminar-09-tasks.md` с формулировками и ноутбук `seminar-09-tasks.ipynb` с заготовками и тестами; они решаются на тех же данных, но по Московской области. Ответы вводятся в тест Moodle «Семинар 9».
- Шпаргалка по функциям: `reference/visualization.md`. Сегодня выдаётся этап 2 проекта, `project/02-eda.md`; разбор его требований — в разделе 7.

## 1. Группы на одном графике: `hue`

На прошлом занятии у графика появился скелет: создать оси, нарисовать, подписать, сохранить. Сегодня на графике появляется третья переменная. Пока на рисунке были две оси, он отвечал на вопрос «сколько» или «когда»; цвет добавляет вопрос «у кого»: тёмное или светлое время, мужчины или женщины, какой год. В seaborn за это отвечает параметр `hue`: имя столбца, по значениям которого раскрашиваются группы.

Подключение то же, что на семинаре 8, плюс `plotly.express` под именем `px` для интерактивных графиков во второй половине занятия. Таблица ДТП Брянской области загружается с датами, и сразу добавляется столбец часа: он нужен в нескольких графиках.

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import seaborn as sns

sns.set_theme(style='whitegrid')                                         # оформление графиков по умолчанию
os.makedirs('img', exist_ok=True)                                        # папка для файлов
dtp = pd.read_csv('../datasets/legal/dtp-bryansk.csv', index_col='id', parse_dates=['datetime'])   # ДТП Брянской области
dtp['hour'] = dtp['datetime'].dt.hour                                    # час каждого ДТП
dtp.shape

В столбце `light` пять значений: светлое время, сумерки и три вида тёмного времени, различающиеся освещением. Для графика их многовато: три вида тёмного времени сводятся к одному словарём через `map`, как на семинаре 6 заменялись коды.

In [ ]:
kinds = {'Светлое время суток': 'светлое', 'Сумерки': 'сумерки',
         'В темное время суток, освещение включено': 'тёмное',
         'В темное время суток, освещение отсутствует': 'тёмное',
         'В темное время суток, освещение не включено': 'тёмное'}
dtp['light_kind'] = dtp['light'].map(kinds)                              # пять значений в три
dtp['light_kind'].value_counts()

Гистограмма часа с семинара 8, но столбец каждого часа разложен на слои по освещению: `hue='light_kind'` задаёт цвет, `multiple='stack'` ставит слои друг на друга, `discrete=True` по-прежнему даёт столбец на каждый час. `hue_order` задаёт порядок слоёв и легенды; освещение упорядочено от светлого к тёмному, поэтому слои раскрашены оттенками одного цвета через `palette`, как тяжесть на семинаре 8; заголовок легенды seaborn берёт из имени столбца, `ax.get_legend().set_title` заменяет его.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(data=dtp, x='hour', hue='light_kind', hue_order=['светлое', 'сумерки', 'тёмное'],
             palette=['#c6dbef', '#6baed6', '#08519c'], multiple='stack', discrete=True, ax=ax)   # слои по освещению
ax.set_title('ДТП по часам суток и освещению')
ax.set_xlabel('Час')
ax.set_ylabel('Число ДТП')
ax.get_legend().set_title('Освещение');                                  # заголовок легенды

Высота столбца та же, что на прошлом занятии: в 18 часов 798 ДТП. Но теперь видно, из чего он состоит: 386 в светлое время, 52 в сумерки и 360 в тёмное. Дальше вечерний пик целиком уходит в темноту: в 21 час из 515 ДТП 430 тёмных, доля 0,83, а в 23 часа 0,95. Один и тот же вопрос «когда» получает второй слой ответа: когда и при каком свете. Без `multiple='stack'` seaborn наложил бы слои друг на друга полупрозрачно, и высоты читались бы неверно.

Вторая форма сравнения групп — плотность вместо гистограммы. Стаж водителей по полу; таблица участников готовится как на семинарах 6 и 8: коды 95, 96 и 99 заменяются пропуском, водители соединяются со своими ДТП.

In [ ]:
people = pd.read_csv('../datasets/legal/dtp-bryansk-participants.csv')   # участники ДТП
people['experience_clean'] = people['experience'].replace([95, 96, 99], np.nan)   # коды стажа пропусками
drivers = people[people['role'] == 'Водитель'].merge(dtp, left_on='dtp_id', right_index=True)   # водители с их ДТП
drivers['gender'].value_counts()

Женщин-водителей 1 916, мужчин 12 752. Две гистограммы такого разного размера на одних осях не сравнить: столбцы женщин утонут внизу. `sns.kdeplot` рисует вместо столбцов сглаженную кривую плотности, а `common_norm=False` нормирует каждую группу отдельно: площадь под каждой кривой равна единице, и сравниваются формы распределений, а не размеры групп.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.kdeplot(data=drivers, x='experience_clean', hue='gender', common_norm=False, ax=ax)   # плотность стажа по полу
ax.set_title('Стаж водителей: мужчины и женщины')
ax.set_xlabel('Стаж, лет')
ax.set_ylabel('Плотность')
ax.get_legend().set_title('Пол');

Кривая женщин высокая и узкая: пик около 4 лет стажа, медиана 7 лет, стаж больше 25 лет лишь у 2 % женщин. Кривая мужчин низкая и широкая: медиана 15 лет, у 25 % стаж больше 25. Высота кривой здесь означает не число водителей, а то, какая доля группы сосредоточена около этого стажа. Без `common_norm=False` seaborn делит одну единицу площади между группами пропорционально их размеру: кривая женщин стала бы в семь раз ниже и выглядела бы плоской.

Ловушка без ошибки, при запасе времени: плотность хороша для непрерывной величины, а для целых значений даёт гладкую кривую там, где значений нет. Плотность часа ДТП:

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.kdeplot(data=dtp, x='hour', ax=ax)                                   # плотность часа: ловушка
ax.set_title('Плотность часа ДТП: так не надо')
ax.set_xlabel('Час');

Кривая тянется от минус трёх до двадцати шести часов, которых не бывает, а ночной минимум и вечерний пик сглажены. Для часов, числа раненых и других целых величин нужна гистограмма с `discrete=True`; плотность — для стажа, возраста, сумм.

## 2. Два числа: рассеяние и корреляция

Семинар 8 обещал связи между числами. Первый инструмент — диаграмма рассеяния: точка на каждую строку, по осям два числа. Связан ли стаж водителя с возрастом его машины? Возраст машины — год ДТП минус год выпуска; `sns.scatterplot` рисует точки, `alpha` делает их полупрозрачными, иначе 13 900 точек сольются в пятно.

In [ ]:
drivers['vehicle_age'] = drivers['datetime'].dt.year - drivers['vehicle_year']   # возраст машины в годах
fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(data=drivers, x='vehicle_age', y='experience_clean', s=10, alpha=0.2, ax=ax)   # точка на водителя
ax.set_title('Стаж водителя и возраст машины')
ax.set_xlabel('Возраст машины, лет')
ax.set_ylabel('Стаж, лет');

Облако без формы: ни наклона, ни сгущения вдоль линии. Число, которое это выражает, — коэффициент корреляции: от −1 при строгой обратной связи до 1 при строгой прямой, около нуля при отсутствии линейной связи. Метод `.corr()` у таблицы из двух столбцов считает его и возвращает матрицу 2 × 2.

In [ ]:
drivers[['vehicle_age', 'experience_clean']].corr()                       # корреляция двух столбцов

Минус 0,03: связи нет. На диагонали единицы, корреляция столбца с самим собой, они ничего не значат. `sns.regplot` добавляет к рассеянию линию тренда (при запасе времени):

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.regplot(data=drivers, x='vehicle_age', y='experience_clean',
            scatter_kws={'s': 10, 'alpha': 0.2}, line_kws={'color': 'black'}, ax=ax)   # точки и линия тренда
ax.set_title('Стаж водителя и возраст машины: линия тренда')
ax.set_xlabel('Возраст машины, лет')
ax.set_ylabel('Стаж, лет');

Линия почти горизонтальна: на каждый год возраста машины минус 0,05 года стажа. Линия тренда не доказывает связь, а описывает её: даже через облако без формы seaborn проведёт прямую. Доказательством связи занимаются семинары 11–13.

Корреляция видит только прямую линию. Сводка по группам видит больше: медиана возраста машины у водителей со стажем до двух лет и у остальных.

In [ ]:
novice = drivers['experience_clean'] <= 2                                # водители со стажем до двух лет
drivers.groupby(novice)['vehicle_age'].median()                          # медиана возраста машины

У новичков машины старше: 13 лет против 10 у остальных. Корреляция −0,03 этой разницы не показывает, потому что она не линейная: разница есть между новичками и всеми остальными, а дальше стаж с возрастом машины не связан. Правило: корреляция около нуля означает «нет прямой линии», а не «нет связи».

Корреляционная матрица — те же коэффициенты для всех пар числовых столбцов сразу. Метод `.corr()` у всей таблицы ДТП падает: в таблице есть текст. Ошибка в следующей ячейке намеренная.

In [ ]:
dtp.corr()                                                               # корреляция по всей таблице: ошибка

Сообщение называет первое текстовое значение, на котором pandas сломался: район. Выход — список числовых столбцов: не все, а те, у которых связь имеет смысл. Координаты из списка исключены: корреляция широты с числом раненых ничего не значит. Матрицу рисует `sns.heatmap`, как тепловую карту двух категорий на семинаре 8, но палитра другая: у корреляции есть знак, поэтому палитра расходящаяся, `coolwarm`, с нейтральным цветом в нуле, `center=0`; `annot=True` с `fmt='.2f'` вписывает коэффициенты в клетки.

In [ ]:
numeric = ['participants_count', 'vehicles_count', 'dead_count', 'injured_count', 'road_defects', 'hour']
corr = dtp[numeric].corr()                                               # матрица корреляций
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr, cmap='coolwarm', center=0, annot=True, fmt='.2f', ax=ax)   # красное плюс, синее минус
ax.set_title('Корреляции числовых столбцов таблицы ДТП');

Участники–раненые 0,68 и участники–транспортные средства 0,50: чем больше участников, тем больше машин и раненых. Погибшие–раненые −0,17: пострадавший либо погиб, либо ранен, и в один счётчик не попадает дважды. Час и недостатки дороги ни с чем не связаны. Корреляция счётчиков между собой ожидаема и ничего не объясняет; интересное на такой карте — неожиданные пары и неожиданные нули. Последовательная палитра `Blues` здесь не годится: у неё нет середины, и −0,17 слился бы с 0,04.

## 3. Сводка внутри графика: доля, среднее и полоса неопределённости

На семинаре 8 сводка считалась отдельно, а график её показывал. seaborn умеет считать сводку сам: `sns.lineplot` по сырой таблице считает среднее `y` для каждого значения `x`. Если `y` — логический столбец, среднее равно доле истинных значений. Доля ДТП с погибшими по часам:

In [ ]:
dtp['fatal'] = dtp['dead_count'] > 0                                     # ДТП с погибшими
fig, ax = plt.subplots(figsize=(8, 4))
sns.lineplot(data=dtp, x='hour', y='fatal', ax=ax)                       # доля по часам с полосой
ax.set_title('Доля ДТП с погибшими по часам суток')
ax.set_xlabel('Час')
ax.set_ylabel('Доля ДТП с погибшими');

Линия — доля: от 0,081 в 9 часов до 0,259 в 1 час ночи. Вокруг линии полоса: 95-процентный доверительный интервал доли, то есть разброс, который давали бы другие выборки того же размера. Ночью полоса широкая: в 1 час ночи всего 158 ДТП, и доля могла бы оказаться где угодно от 0,20 до 0,32. В 18 часов ДТП 798, полоса от 0,11 до 0,16. Урок семинара 5 о малых группах стал видимым: чем меньше группа, тем шире полоса, и тем меньше стоит верить разнице между соседними точками.

То же со столбцами: `sns.barplot` считает среднее `y` по группам `x` и рисует усы того же интервала. Среднее число погибших на ДТП по числу участников; ДТП с девятью и более участниками мало, 15 при девяти и от одного до шести дальше, их отбрасываем.

In [ ]:
few = dtp[dtp['participants_count'] <= 8]                                # ДТП до восьми участников
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=few, x='participants_count', y='dead_count', ax=ax)     # среднее погибших с усами
ax.set_title('Погибших на одно ДТП по числу участников')
ax.set_xlabel('Участников')
ax.set_ylabel('Погибших на ДТП');

Среднее растёт с 0,14 при одном участнике до 0,70 при восьми. Усы растут вместе с ним: при двух участниках 5 914 ДТП и усы едва видны, при восьми участниках 23 ДТП и усы длиннее столбца. Столбец без усов сообщил бы «0,70», столбец с усами сообщает «где-то между 0,4 и 1,0, точнее по 23 ДТП не скажешь».

## 4. Много линий и много панелей

Линия по месяцам за все годы прятала годы внутри суммы. `hue` раскладывает её на линии по годам: сводка год × месяц через `groupby` по двум столбцам, `size` и `reset_index(name='count')`, чтобы год и месяц стали столбцами, а не индексом. Годы упорядочены, поэтому палитра последовательная, `viridis`: ранние годы светлые, поздние тёмные; `legend='full'` показывает все одиннадцать лет.

In [ ]:
dtp['year'] = dtp['datetime'].dt.year
dtp['month'] = dtp['datetime'].dt.month
monthly = dtp.groupby(['year', 'month']).size().reset_index(name='count')   # ДТП по годам и месяцам
fig, ax = plt.subplots(figsize=(9, 5))
sns.lineplot(data=monthly, x='month', y='count', hue='year', palette='viridis', legend='full', ax=ax)   # линия на год
ax.set_title('ДТП по месяцам в каждый год')
ax.set_xlabel('Месяц')
ax.set_ylabel('Число ДТП')
ax.get_legend().set_title('Год');

Одиннадцать линий, и порядок лет виден по цвету без чтения легенды: светлые линии 2015–2018 годов вверху, тёмные 2024–2025 внизу. Сезонность одна и та же: летний подъём и зимний спад; уровень падает год от года. Самый аварийный месяц — август 2018 года, 143 ДТП; самый тихий — январь 2024-го, 20. Провал в апреле 2020 года, 45 ДТП против 62–94 в соседние месяцы того же года, совпадает с карантином той весны.

Когда групп много, вместо одного графика с десятком цветов лучше панели: по графику на группу. `sns.displot` строит их одной командой: `col='weekday'` даёт панель на каждый день недели, `col_wrap=4` переносит панели на новый ряд после четырёх, `height` задаёт высоту панели. Результат — не оси, а объект `FacetGrid`; фигура лежит в `g.figure`, и общий заголовок ставится ей. `set_titles` и `set_axis_labels` подписывают все панели сразу.

In [ ]:
dtp['weekday'] = dtp['datetime'].dt.dayofweek                            # понедельник 0
g = sns.displot(data=dtp, x='hour', col='weekday', col_wrap=4, discrete=True, height=2.5)   # панель на день
g.set_titles('День недели {col_name}')
g.set_axis_labels('Час', 'Число ДТП')
g.figure.suptitle('ДТП по часам в каждый день недели, понедельник 0')
g.figure.tight_layout();

Семь гистограмм на одной шкале: сравнивать можно и высоты, и форму. Пик рабочих дней в 18 часов, во вторник в 17, в четверг и воскресенье в 19; самый высокий пик у пятницы, 129 ДТП. Общая шкала по оси Y честна: воскресенье ниже не потому, что панель меньше, а потому, что ДТП меньше.

Панели по двум признакам, при запасе времени: `sns.catplot` строит ящики стажа по тяжести ДТП отдельно для мужчин и женщин; `kind='box'` выбирает вид графика, `col='gender'` раскладывает по панелям.

In [ ]:
order = ['Легкий', 'Тяжёлый', 'С погибшими']
g = sns.catplot(data=drivers, x='severity', y='experience_clean', kind='box', order=order,
                col='gender', height=4, aspect=1.1)                      # ящики по полу
g.set_titles('{col_name} пол')
g.set_axis_labels('Тяжесть ДТП', 'Стаж, лет')
g.figure.suptitle('Стаж водителей по тяжести ДТП')
g.figure.tight_layout();

У женщин медианы 7, 7 и 7 лет, у мужчин 14, 15 и 15: тяжесть ДТП от стажа не зависит ни у тех, ни у других, а разница между панелями — та же, что на кривых плотности.

`sns.pairplot` рисует рассеяние каждой пары числовых столбцов и гистограмму каждого столбца на диагонали, при запасе времени; строки с пропусками он не принимает, поэтому `dropna`.

In [ ]:
three = drivers[['experience_clean', 'vehicle_age', 'violations_count']].dropna()   # три числовых столбца
g = sns.pairplot(three, plot_kws={'s': 8, 'alpha': 0.2}, height=2.5)     # все пары столбцов
g.figure.suptitle('Стаж, возраст машины и число нарушений')
g.figure.tight_layout();

Все облака без формы: ни одна пара не связана. Для отчёта такая сетка — быстрая разведка, какие пары стоит рисовать по отдельности; здесь не стоит ни одну.

## 5. Plotly: график, который отвечает на наведение

Графики matplotlib и seaborn — картинки. Plotly рисует графики, которые живут в браузере: при наведении на точку всплывает подсказка с её значениями, колёсико меняет масштаб, щелчок по легенде прячет линию. `plotly.express`, `px`, строит такие графики из таблицы одной функцией. Та же сводка по годам и месяцам: `color='year'` раскладывает линии по годам, `markers=True` ставит точки, `title` даёт заголовок. Подписи осей задаёт `fig.update_layout`. Фигура показывается, когда она последняя строка ячейки, либо через `fig.show()`.

In [ ]:
fig = px.line(monthly, x='month', y='count', color='year', markers=True, title='ДТП по месяцам и годам')   # интерактивные линии
fig.update_layout(xaxis_title='Месяц', yaxis_title='Число ДТП')
fig

Наведите на точку: месяц, год и число ДТП в подсказке, читать значения по сетке не нужно. Щёлкните по году в легенде: линия спрячется; двойной щелчок оставит один год. Фигура plotly — объект с двумя частями: `fig.data` хранит линии и их точки, `fig.layout` — заголовок, оси и легенду; так её читают и тесты упражнений.

Столбцы с подсказкой, при запасе времени: сводка `value_counts` превращается в таблицу с двумя столбцами, `px.bar` рисует её; `orientation='h'` кладёт столбцы горизонтально.

In [ ]:
top6 = dtp['category'].value_counts().head(6).reset_index()               # таблица: вид, число
fig = px.bar(top6, x='count', y='category', orientation='h', title='Шесть самых частых видов ДТП')   # столбцы с подсказкой
fig.update_layout(xaxis_title='Число ДТП', yaxis_title='', yaxis={'categoryorder': 'total ascending'})
fig

Интерактивный график сохраняется не в PNG, а в HTML: файл открывается в браузере без Python, и подсказки в нём работают. По умолчанию `write_html` встраивает в файл всю библиотеку plotly, около пяти мегабайт; `include_plotlyjs='cdn'` подключает её из интернета, и файл весит килобайты. На GitHub интерактивные фигуры не показываются, в PDF не печатаются: для отчёта на бумаге нужен matplotlib.

In [ ]:
fig = px.line(monthly, x='month', y='count', color='year', markers=True, title='ДТП по месяцам и годам')
fig.write_html('img/dtp-by-month.html', include_plotlyjs='cdn')          # файл для браузера
os.path.getsize('img/dtp-by-month.html')

Откройте `notebooks/img/dtp-by-month.html` двойным щелчком: тот же график в браузере.

## 6. Карта

Второе обещание семинара 8 — карта с подложкой. Точки берутся внутри рамки области, как на прошлом занятии; `px.scatter_map` ставит их на карту: `lat` и `lon` — столбцы координат, `color='severity'` красит точки по тяжести, `hover_data` добавляет в подсказку район и дату, `zoom` — начальный масштаб. Цвета групп задаёт `color_discrete_map`: без него plotly раздаёт цвета по порядку появления, и ДТП с погибшими достались бы зелёные. Подложка карты грузится из интернета; без сети она не загрузится.

In [ ]:
inside = dtp[dtp['lat'].between(51.8, 54.2) & dtp['lon'].between(31, 35.5)]   # ДТП внутри рамки области
colors = {'Легкий': '#a6cee3', 'Тяжёлый': '#1f78b4', 'С погибшими': '#e31a1c', 'Без пострадавших (нет данных)': '#999999'}
fig = px.scatter_map(inside, lat='lat', lon='lon', color='severity', color_discrete_map=colors,
                     hover_data=['district', 'datetime'], zoom=7, height=600, title='ДТП Брянской области')   # точки на карте
fig

На карте 9 747 ДТП, из них 1 314 с погибшими красным. Приблизьте колёсиком Брянск: точки ложатся на улицы. Щёлкните в легенде по «Тяжёлый» и «Легкий», и останутся только ДТП с погибшими: по трассам они идут цепочками. Наведите на точку: район и дата.

Ловушка без ошибки: координаты перепутаны местами, `lat='lon', lon='lat'`. Ошибки не будет: числа как числа.

In [ ]:
fig = px.scatter_map(inside, lat='lon', lon='lat', zoom=3, height=400, title='Широта и долгота перепутаны')   # ловушка
fig

Карта центрируется на 34° северной широты и 53° восточной долготы: ДТП Брянской области уехали в Иран. На карте такая ошибка видна с первого взгляда, в таблице чисел — нет; на прошлом занятии то же показывало рассеяние без подложки.

## 7. Этап 2 проекта: из каких графиков состоит отчёт

Сегодня выдаётся этап 2 проекта, разведочный анализ своего датасета: `project/02-eda.md`, срок — семинар 11. Его разделы требуют графиков, и все они уже есть в курсе как функции части Б семинаров 8 и 9:

| Раздел этапа 2 | Что нарисовать | Чем |
|---|---|---|
| Пропуски | доли пропусков по столбцам | столбцы долей, задача В2 семинара 9 |
| Распределения числовых столбцов | гистограмма целых значений, ящик по группам, плотность по группам | `count_hist`, `box_by` семинара 8; `density_by` семинара 9 |
| Категории | самые частые значения столбцами, состав по группам | `top_bars`, `share_bars` семинара 8; `stacked_hist` семинара 9 |
| Время | число по периодам, доля признака по периодам, линии по годам | `count_line` семинара 8; `share_line`, `count_lines` семинара 9 |
| Корреляции | матрица числовых столбцов | `corr_heat` семинара 9 |
| Связи | рассеяние, среднее по группам с усами, столбцы рядом, панели | `scatterplot`; `mean_bars`, `hist_panels` семинара 9; `grouped_bars` семинара 8 |

Графиков нужно не меньше пяти, разных, с подписями и выводом под каждым; интерактивный график plotly даёт бонус. К каждому графику — памятка из пяти вопросов с прошлого занятия: что показывает, что по осям, тот ли тип, порядок и масштаб, какой вывод. Необязательная задача сегодняшнего семинара — пять графиков по своему датасету — и есть черновик раздела «Визуализация» этапа 2.

## Итоги

Третья переменная на графике задаётся цветом: `hue` в `histplot` с `multiple='stack'`, в `kdeplot` с `common_norm=False`, в `lineplot` с последовательной палитрой для упорядоченных групп. Связь двух чисел показывает `scatterplot`, измеряет `.corr()`, описывает `regplot`; матрицу корреляций рисует `heatmap` с расходящейся палитрой и `center=0`, а `.corr()` по всей таблице с текстом падает. `lineplot` и `barplot` считают среднее сами и рисуют полосу или усы доверительного интервала: по ним видно, где группа мала. Панели одной командой дают `displot` и `catplot`, фигура лежит в `g.figure`. Plotly строит интерактивные графики из таблицы: `px.line`, `px.bar`, `px.scatter_map`; подписи через `fig.update_layout`, файл через `fig.write_html` с `include_plotlyjs='cdn'`. Корреляция около нуля означает «нет прямой линии», а не «нет связи»; перепутанные координаты видны на карте сразу.

Переходите к упражнениям: откройте `exercises/seminar-09-tasks.ipynb`, формулировки в `exercises/seminar-09-tasks.md`; данные там по Московской области. Функции части Б возвращают оси, фигуру или фигуру plotly; в части В из них собирается разворот отчёта. Этап 2 проекта выдан сегодня, срок — семинар 11; необязательная задача — пять графиков по своему датасету — его черновик.